# Imports

In [1]:
from math import log as log
from math import inf as inf
import numpy as np

# Constantes Globais

In [2]:
p = 0.5
N = 500 #definido arbitrariamente
MAXLEVEL = int(log(N, 1/p))

# TASKs

In [3]:
class Task:
    def __init__(self, atributos: dict):
        #att = ["task_id", "start_time", "end_times"]
        self.atributos = atributos

    def __getitem__(self, chave):
        return self.atributos[chave]

    def __setitem__(self, chave, valor):
        self.atributos[chave] = valor

# Skip List - ATIVIDADE B)

In [4]:
class No_skiplist:
    def __init__(self, nivel:int=0, tasks:Task = None):
        self.nivel = nivel
        self.frente = [None] * nivel
        self.conteudo = tasks

In [5]:
class NULL:
  def __init__(self):
    self.nivel = MAXLEVEL
    self.conteudo = {"task_id":inf, "start_time": inf, "end_time": inf}

In [6]:
class Skip_list:
  def __init__(self):
    self.tam = 0
    self.nivel_atual = 0
    self.NULL = NULL()
    self.cabeca = No_skiplist(nivel = MAXLEVEL,
                              tasks={"task_id":-inf, "start_time": -inf, "end_time": -inf})
    for i in range(len(self.cabeca.frente)):
      self.cabeca.frente[i] = self.NULL

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================

  def insert(self, chaveBusca: int, novoStartTime, novoEndTime) -> None:
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x != self.NULL and x.conteudo['task_id'] == chaveBusca:
      x.conteudo['start_time'] = novoStartTime
      x.conteudo['end_time'] = novoEndTime

    else:
      nvl = 0
      while np.random.random() < p and nvl < MAXLEVEL-1:
        nvl+=1

      if nvl > self.nivel_atual:
        for i in range(self.nivel_atual+1, nvl+1):
          atualiza[i] = self.cabeca
        self.nivel_atual = nvl

      task = Task({"task_id": chaveBusca, "start_time":
                   novoStartTime, "end_time": novoEndTime})

      x = No_skiplist(nvl+1, task)
      for i in range(x.nivel):
        x.frente[i] = self.NULL

      for i in range(nvl+1):
        x.frente[i] = atualiza[i].frente[i]
        atualiza[i].frente[i] = x

      self.tam+=1

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, chaveBusca):
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x!=self.NULL and x.conteudo['task_id'] == chaveBusca:
      for i in range(self.nivel_atual+1):
        if atualiza[i].frente[i] != x:
          break
        atualiza[i].frente[i] = x.frente[i]

      self.tam-=1

    while self.nivel_atual > 0 and self.cabeca.frente[self.nivel_atual] == self.NULL:
      self.nivel_atual-=1

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, chaveBusca):
    x = self.cabeca
    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]

    x = x.frente[0]
    if x.conteudo['task_id'] == chaveBusca:
      return True
    else:
      return False
#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self,chaveBuscaA, chaveBuscaB):
    x = self.cabeca
    y = self.cabeca
    output = [None]*(self.tam+1)
    nvly = self.nivel_atual

    for i in range(self.nivel_atual, -1, -1):
      while y.frente[i] != self.NULL and y.frente[i].conteudo['task_id'] <= chaveBuscaB:
        y = y.frente[i]
      nvly = i

    for j in range(nvly,-1,-1):
      while x.frente[j] != self.NULL and x.frente[j].conteudo['task_id'] < chaveBuscaA:
        x = x.frente[j]

    x = x.frente[0]
    c = 0

    if x.conteudo['task_id'] >= chaveBuscaA and y.conteudo['task_id'] <= chaveBuscaB:
      while x.conteudo['task_id'] <= y.conteudo['task_id']:
        output[c] = x.conteudo
        x = x.frente[0]
        c+=1

    return output

# EXEMPLO DE USO - SKIP LIST

In [7]:
skip = Skip_list()

## TESTE INSERT

In [8]:
#c = ["task_id", "start_time", "end_times"]
for i in range(MAXLEVEL):
  x = np.random.randint(0,100)
  y = np.random.randint(0,1000)
  z = np.random.randint(0,2000)
  # t = t = dict(zip(c, [x, y, z]))
  t = Task({"task_id":x,"start_time":y, 'end_time': z})
  skip.insert(x,y,z)


### VISUALIZAÇÃO SKIPLIST

In [9]:
x = skip.cabeca.frente[0]

while x != skip.NULL:
    print(x.conteudo.atributos)
    x = x.frente[0]

{'task_id': 3, 'start_time': 240, 'end_time': 834}
{'task_id': 6, 'start_time': 546, 'end_time': 1435}
{'task_id': 7, 'start_time': 723, 'end_time': 1463}
{'task_id': 24, 'start_time': 333, 'end_time': 450}
{'task_id': 31, 'start_time': 451, 'end_time': 990}
{'task_id': 73, 'start_time': 530, 'end_time': 233}
{'task_id': 75, 'start_time': 114, 'end_time': 1898}


In [10]:
for i in range(skip.nivel_atual, -1, -1):
    print(f"--- Nível {i} ---")
    x = skip.cabeca.frente[i]

    while x != skip.NULL:
        print(x.conteudo.atributos['task_id'], end=" -> ")
        x = x.frente[i]
    print("NULL")

--- Nível 3 ---
31 -> 75 -> NULL
--- Nível 2 ---
24 -> 31 -> 75 -> NULL
--- Nível 1 ---
3 -> 7 -> 24 -> 31 -> 73 -> 75 -> NULL
--- Nível 0 ---
3 -> 6 -> 7 -> 24 -> 31 -> 73 -> 75 -> NULL


## TESTE DELETE

In [11]:
skip.delete(5)

### VISUALIZAÇÃO SKIPLIST

In [12]:
x = skip.cabeca.frente[0]

while x != skip.NULL:
    print(x.conteudo.atributos)
    x = x.frente[0]

{'task_id': 3, 'start_time': 240, 'end_time': 834}
{'task_id': 6, 'start_time': 546, 'end_time': 1435}
{'task_id': 7, 'start_time': 723, 'end_time': 1463}
{'task_id': 24, 'start_time': 333, 'end_time': 450}
{'task_id': 31, 'start_time': 451, 'end_time': 990}
{'task_id': 73, 'start_time': 530, 'end_time': 233}
{'task_id': 75, 'start_time': 114, 'end_time': 1898}


In [13]:
for i in range(skip.nivel_atual, -1, -1):
    print(f"--- Nível {i} ---")
    x = skip.cabeca.frente[i]

    while x != skip.NULL:
        print(x.conteudo.atributos['task_id'], end=" -> ")
        x = x.frente[i]
    print("NULL")

--- Nível 3 ---
31 -> 75 -> NULL
--- Nível 2 ---
24 -> 31 -> 75 -> NULL
--- Nível 1 ---
3 -> 7 -> 24 -> 31 -> 73 -> 75 -> NULL
--- Nível 0 ---
3 -> 6 -> 7 -> 24 -> 31 -> 73 -> 75 -> NULL


## Teste SEARCH

In [14]:
skip.search(26)

False

### VISUALIZAÇÃO SKIPLIST

In [15]:
x = skip.cabeca.frente[0]

while x != skip.NULL:
    print(x.conteudo.atributos)
    x = x.frente[0]

{'task_id': 3, 'start_time': 240, 'end_time': 834}
{'task_id': 6, 'start_time': 546, 'end_time': 1435}
{'task_id': 7, 'start_time': 723, 'end_time': 1463}
{'task_id': 24, 'start_time': 333, 'end_time': 450}
{'task_id': 31, 'start_time': 451, 'end_time': 990}
{'task_id': 73, 'start_time': 530, 'end_time': 233}
{'task_id': 75, 'start_time': 114, 'end_time': 1898}


In [16]:
for i in range(skip.nivel_atual, -1, -1):
    print(f"--- Nível {i} ---")
    x = skip.cabeca.frente[i]

    while x != skip.NULL:
        print(x.conteudo.atributos['task_id'], end=" -> ")
        x = x.frente[i]
    print("NULL")

--- Nível 3 ---
31 -> 75 -> NULL
--- Nível 2 ---
24 -> 31 -> 75 -> NULL
--- Nível 1 ---
3 -> 7 -> 24 -> 31 -> 73 -> 75 -> NULL
--- Nível 0 ---
3 -> 6 -> 7 -> 24 -> 31 -> 73 -> 75 -> NULL


## Teste SEARCHINTERVAL

In [17]:
r = skip.searchinterval(17,88)

In [18]:
for i in range(len(r)):
  if r[i] != None:
    print(r[i].atributos)

{'task_id': 24, 'start_time': 333, 'end_time': 450}
{'task_id': 31, 'start_time': 451, 'end_time': 990}
{'task_id': 73, 'start_time': 530, 'end_time': 233}
{'task_id': 75, 'start_time': 114, 'end_time': 1898}


### VISUALIZAÇÃO SKIPLIST

In [19]:
x = skip.cabeca.frente[0]

while x != skip.NULL:
    print(x.conteudo.atributos)
    x = x.frente[0]

{'task_id': 3, 'start_time': 240, 'end_time': 834}
{'task_id': 6, 'start_time': 546, 'end_time': 1435}
{'task_id': 7, 'start_time': 723, 'end_time': 1463}
{'task_id': 24, 'start_time': 333, 'end_time': 450}
{'task_id': 31, 'start_time': 451, 'end_time': 990}
{'task_id': 73, 'start_time': 530, 'end_time': 233}
{'task_id': 75, 'start_time': 114, 'end_time': 1898}


In [20]:
for i in range(skip.nivel_atual, -1, -1):
    print(f"--- Nível {i} ---")
    x = skip.cabeca.frente[i]

    while x != skip.NULL:
        print(x.conteudo.atributos['task_id'], end=" -> ")
        x = x.frente[i]
    print("NULL")

--- Nível 3 ---
31 -> 75 -> NULL
--- Nível 2 ---
24 -> 31 -> 75 -> NULL
--- Nível 1 ---
3 -> 7 -> 24 -> 31 -> 73 -> 75 -> NULL
--- Nível 0 ---
3 -> 6 -> 7 -> 24 -> 31 -> 73 -> 75 -> NULL


# AVL - ATIVIDADE C)

In [21]:
class No_AVL:
  def __init__(self, esq=None, dir=None, task:Task=None):
    self.esq = esq
    self.dir = dir
    self.fator_balanceamento = 0
    self.altura = -1
    self.conteudo = task

In [22]:
class AVL:
  def __init__(self):
      pass

  def MAX(self, h_esq, h_dir):
    if h_dir >= h_esq:
      return h_dir

    else:
      return h_esq


  def calculaAltura(self, raiz:No_AVL):
    if raiz == None:
      return -1

    h_esq = self.calculaAltura(raiz.esq)
    h_dir = self.calculaAltura(raiz.dir)

    return 1 + self.MAX(h_esq, h_dir)


  def calculaFatorBalanceamento(self, raiz:No_AVL):
    if raiz == None:
      return 0
    return raiz.esq.altura - raiz.dir.altura

  def rotacaoESQUERDA(self, raiz:No_AVL):
    n_aux = No_AVL()
    t_aux = None

    n_aux = raiz.dir
    t_aux = n_aux.esq

    n_aux.esq = raiz
    raiz.dir = t_aux

    raiz.altura = 1 + self.MAX(raiz.esq.altura, raiz.dir.altura)
    n_aux.altura = 1 + self.MAX(raiz.esq.altura, raiz.dir.altura)

    return n_aux

  def rotacaoDIREITA(self,raiz:No_AVL):
    n_aux = No_AVL()
    t_aux = None

    n_aux = raiz.esq
    t_aux = n_aux.dir

    n_aux.dir = raiz
    raiz.esq = t_aux

    raiz.altura = 1 + self.MAX(raiz.esq.altura, raiz.dir.altura)
    n_aux.altura = 1 + self.MAX(raiz.esq.altura, raiz.dir.altura)

    return n_aux


  def insert(self, raiz:No_AVL, task:Task):
    if raiz == None:
      return No_AVL(None, None, task)

    if task.atributos['task_id'] < raiz.conteudo['task_id']:
      raiz.esq = self.insert(raiz.esq, task)

    else:
      raiz.dir = self.insert(raiz.dir, task)

    raiz.altura = self.calculaAltura(raiz)
    raiz.fator_balanceamento = self.calculaFatorBalanceamento(raiz)

    #LL
    if raiz.fator_balanceamento > 1 and raiz.esq.fator_balanceamento >= 0:
      return self.rotacaoDIREITA(raiz)

    #RR
    if raiz.fator_balanceamento < -1 and raiz.dir.fator_balanceamento <= 0:
      return self.rotacaoESQUERDA(raiz)

    #LR
    if raiz.fator_balanceamento > 1 and raiz.esq.fator_balanceamento < 0:
      raiz.esq = self.rotacaoESQUERDA(raiz.esq)
      return self.rotacaoDIREITA(raiz)

    #RL
    if raiz.fator_balanceamento < -1 and raiz.esq.fator_balanceamento > 0:
      raiz.dir = self.rotacaoDIREITA(raiz.dir)
      return self.rotacaoESQUERDA(raiz)

    return raiz

  def antecessor(self, raiz:No_AVL):
    atual = raiz
    while atual.dir != None:
      atual = atual.dir
    return atual

  def delete(self, raiz:No_AVL, chaveBusca):
    if raiz == None:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      raiz.esq = self.delete(raiz.esq)

    elif chaveBusca > raiz.conteudo['task_id']:
      raiz.dir = self.delete(raiz.dir)

    else:
      if raiz.esq == None:
        aux = raiz.dir
        raiz = None
        return aux

      elif raiz.dir == None:
        aux = raiz.esq
        raiz = None
        return aux

      aux = self.antecessor(raiz.esq)
      raiz.conteudo = aux.conteudo
      raiz.esq = self.delete(raiz.esq, aux.conteudo['task_id'])

    if raiz == None:
      return raiz

    raiz.altura = self.calculaAltura(raiz)
    raiz.fator_balanceamento = self.calculaFatorBalanceamento(raiz)

    #LL
    if raiz.fator_balanceamento > 1 and raiz.esq.fator_balanceamento >= 0:
      return self.rotacaoDIREITA(raiz)

    #RR
    if raiz.fator_balanceamento < -1 and raiz.dir.fator_balanceamento <= 0:
      return self.rotacaoESQUERDA(raiz)

    #LR
    if raiz.fator_balanceamento > 1 and raiz.esq.fator_balanceamento < 0:
      raiz.esq = self.rotacaoESQUERDA(raiz.esq)
      return self.rotacaoDIREITA(raiz)

    #RL
    if raiz.fator_balanceamento < -1 and raiz.esq.fator_balanceamento > 0:
      raiz.dir = self.rotacaoDIREITA(raiz.dir)
      return self.rotacaoESQUERDA(raiz)

    return raiz

  def search(self, raiz:No_AVL, chaveBusca):
    if raiz == None or raiz.conteudo['task_id'] == chaveBusca:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      raiz.esq = self.search(raiz.esq)

    elif chaveBusca > raiz.conteudo['task_id']:
      raiz.dir = self.search(raiz.dir)

  def percursoIN_ORDER(self, output:list, raiz:No_AVL):
    if raiz == None:
      return

    self.percursoIN_ORDER(raiz.esq)
    output.append(raiz)
    self.percursoIN_ORDER(raiz.dir)

  def searchinterval(self, raiz:No_AVL, chaveBuscaA, chaveBuscaB):
    if raiz == None:
      return
    #output = [None]*N
    x = raiz
    #y = raiz

    if chaveBuscaA < x.conteudo['task_id']:
      x.esq = self.searchinterval(x.esq, chaveBuscaA, None)
    elif chaveBuscaA > x.conteudo['task_id']:
      x.dir = self.searchinterval(x.dir, chaveBuscaA, None)
#EM X PRECISO DE TODOS OS NÓS MAIORES OU IGUAIS CHAVEBUSCA-A


# EXEMPLO DE USO - AVL